In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [ ]:
# 데이터 불러오기
abalone_df = pd.read_csv(
    'https://storage.googleapis.com/download.tensorflow.org/data/abalone_train.csv',
    names=['Length', 'Diameter', 'Height', 'Whole weight', 'Shucked weight',
           'Viscera weight', 'Shell weight', 'Age']
)

# 입력과 타깃 나누기
input_data = abalone_df.drop(columns=['Age']).to_numpy().astype(np.float32)
target_data = abalone_df['Age'].to_numpy().astype(np.float32)

# 데이터셋 클래스 정의
class AbaloneDataset(Dataset):
    def __init__(self, input_data, target_data):
        self.input_data = input_data
        self.target_data = target_data

    def __len__(self):
        return len(self.input_data)

    def __getitem__(self, index):
        input_tensor = torch.tensor(self.input_data[index])
        target_tensor = torch.tensor(self.target_data[index])

        return input_tensor, target_tensor

# 학습/검증/테스트 분할
train_size = int(len(input_data) * 0.8)
val_size = int(len(input_data) * 0.1)

train_inputs = input_data[:train_size]
train_targets = target_data[:train_size]

val_inputs = input_data[train_size:train_size + val_size]
val_targets = target_data[train_size:train_size + val_size]

test_inputs = input_data[train_size + val_size:]
test_targets = target_data[train_size + val_size:]

# 표준화
scaler = StandardScaler()
scaler.fit(train_inputs)

train_inputs_scaled = scaler.transform(train_inputs)
val_inputs_scaled = scaler.transform(val_inputs)
test_inputs_scaled = scaler.transform(test_inputs)

# 데이터셋 만들기
train_dataset = AbaloneDataset(train_inputs_scaled, train_targets)
val_dataset = AbaloneDataset(val_inputs_scaled, val_targets)
test_dataset = AbaloneDataset(test_inputs_scaled, test_targets)

# 데이터로더 만들기
train_dataloader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=32)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=32)

In [ ]:
# 모델 클래스 정의
class AbaloneModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(7, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 8)
        self.fc4 = nn.Linear(8, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.dropout(x)
        x = self.fc4(x)
        return x


# 모델, 손실 함수, 옵티마이저 객체 만들기
model = AbaloneModel()
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters())

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# training loop
epochs = 10
step = 0
for epoch in range(epochs):
    model.train()
    for train_batch in train_dataloader:
        x_train, y_train = train_batch[0].to(device), train_batch[1].to(device)
        pred = model(x_train).squeeze()
        loss = loss_fn(pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        step += 1

        if step % 100 == 0:
            print(f'step {step}, train loss: {loss.item():.4f}')

    model.eval()
    with torch.no_grad():
        losses = []
        for val_batch in val_dataloader:
            x_val, y_val = val_batch[0].to(device), val_batch[1].to(device)
            pred_val = model(x_val).squeeze()
            loss = loss_fn(pred_val, y_val)
            losses.append(loss.item())

        val_loss_avg = sum(losses) / len(losses)
        print(f'epoch {epoch+1}/{epochs}, validation loss: {val_loss_avg:.4f}\n')

    #모델이 끝날 때마다 저장하도록 함
    torch.save(model.state_dict(), f'model_{epoch + 1}.pth')

epoch 1/10, validation loss: 36.4123

step 100, train loss: 35.1186
epoch 2/10, validation loss: 10.0429

step 200, train loss: 15.6673
epoch 3/10, validation loss: 6.8445

step 300, train loss: 19.9372
epoch 4/10, validation loss: 6.4824

step 400, train loss: 30.2098
epoch 5/10, validation loss: 6.0812

epoch 6/10, validation loss: 7.1363

step 500, train loss: 13.0945
epoch 7/10, validation loss: 6.1022

step 600, train loss: 13.7548
epoch 8/10, validation loss: 6.3252

step 700, train loss: 14.6443
epoch 9/10, validation loss: 5.0935

step 800, train loss: 9.6086
epoch 10/10, validation loss: 5.7751



In [ ]:
# 모델 저장 : model.pth 경로로 모델 저장
torch.save(model.state_dict(), 'model.pth')

In [ ]:
'''
save함수에서 모델이 state_dict 메소드 결과를 넣어주기 때문에
model 파일에는 파라미터 이름과 값만 저장
'''
state_dict = model.state_dict()
state_dict

OrderedDict([('fc1.weight',
              tensor([[-0.1652,  0.1320, -0.1675, -0.0635, -0.2633, -0.1518, -0.1410],
                      [-0.2597, -0.1585,  0.0557, -0.2326,  0.1282, -0.0243,  0.2179],
                      [-0.1346, -0.2941, -0.1052, -0.3697,  0.2071,  0.1787, -0.0703],
                      [ 0.2935, -0.2402, -0.3054,  0.1098, -0.3343, -0.1484,  0.2953],
                      [-0.2066,  0.3175,  0.3714,  0.2129,  0.3545, -0.0009, -0.2510],
                      [-0.3355, -0.0831, -0.0781, -0.1113, -0.0645, -0.3059, -0.3024],
                      [ 0.2066, -0.1525, -0.0844, -0.3180,  0.1151, -0.0888,  0.3513],
                      [ 0.2694, -0.0270, -0.3362, -0.1285,  0.0607,  0.2101, -0.2527],
                      [ 0.2361,  0.1231,  0.3523, -0.0382,  0.2191,  0.2657,  0.1771],
                      [ 0.0495,  0.0794,  0.0967,  0.1520, -0.1649, -0.0644,  0.3215],
                      [-0.2570, -0.2718, -0.3012, -0.1760,  0.2700,  0.0398,  0.3933],
               

In [ ]:
# 모델 불러오기
state_dict_loaded = torch.load('model.pth', weights_only=True)
'''
weights_only가 True면 model 파일에 담긴 파라미터 정보만 불러옴
현재는 파라미터 정보만 담겨져있지만 불필요한 정보를 미리 필터링하는 안전장치
'''
state_dict_loaded

OrderedDict([('fc1.weight',
              tensor([[-0.1652,  0.1320, -0.1675, -0.0635, -0.2633, -0.1518, -0.1410],
                      [-0.2597, -0.1585,  0.0557, -0.2326,  0.1282, -0.0243,  0.2179],
                      [-0.1346, -0.2941, -0.1052, -0.3697,  0.2071,  0.1787, -0.0703],
                      [ 0.2935, -0.2402, -0.3054,  0.1098, -0.3343, -0.1484,  0.2953],
                      [-0.2066,  0.3175,  0.3714,  0.2129,  0.3545, -0.0009, -0.2510],
                      [-0.3355, -0.0831, -0.0781, -0.1113, -0.0645, -0.3059, -0.3024],
                      [ 0.2066, -0.1525, -0.0844, -0.3180,  0.1151, -0.0888,  0.3513],
                      [ 0.2694, -0.0270, -0.3362, -0.1285,  0.0607,  0.2101, -0.2527],
                      [ 0.2361,  0.1231,  0.3523, -0.0382,  0.2191,  0.2657,  0.1771],
                      [ 0.0495,  0.0794,  0.0967,  0.1520, -0.1649, -0.0644,  0.3215],
                      [-0.2570, -0.2718, -0.3012, -0.1760,  0.2700,  0.0398,  0.3933],
               

In [ ]:
print(state_dict.keys())
print(state_dict_loaded.keys())

odict_keys(['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias', 'fc3.weight', 'fc3.bias', 'fc4.weight', 'fc4.bias'])
odict_keys(['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias', 'fc3.weight', 'fc3.bias', 'fc4.weight', 'fc4.bias'])


In [ ]:
for key in state_dict.keys():
    if torch.equal(state_dict[key], state_dict_loaded[key]):
        print(f'{key} 일치')

fc1.weight 일치
fc1.bias 일치
fc2.weight 일치
fc2.bias 일치
fc3.weight 일치
fc3.bias 일치
fc4.weight 일치
fc4.bias 일치


In [ ]:
model_loaded = AbaloneModel()
model_loaded.load_state_dict(state_dict_loaded)

<All keys matched successfully>

In [ ]:
model_loaded.to(device)
model.eval()
model_loaded.eval()


# 기존의 model 객체와 새로운 model_loaded에 같은 텐서 입력 했을 때 결과 같은지 확인
for test_batch in test_dataloader:
    x_test, y_test = test_batch[0].to(device), test_batch[1].to(device)
    pred_org = model(x_test)
    pred_loaded = model_loaded(x_test)

    print(torch.equal(pred_org, pred_loaded))
    break

True
